# 02-attention-visualizer
## Self-Attention Explained End-to-End (Notebook Style)

**Goal:** Understand the complete self-attention mechanism used in Transformer models with a tiny example and manually computed values.

We will use the sentence:

> **"I love AI"**

and calculate:
1. Token embeddings
2. Query (Q), Key (K), and Value (V)
3. Attention scores
4. Scaled dot-product attention
5. Softmax normalization
6. Final attention output

At the end, you should be able to explain the formula

\[
Attention(Q,K,V)=Softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

in interviews and implement it from scratch.


## Step 1: The Sentence

Sentence:

```text
I love AI
```

After tokenization:

| Position | Token |
|----------|--------|
| 1 | I |
| 2 | love |
| 3 | AI |

For learning purposes, we will assume each token has a 2-dimensional embedding.


In [ ]:
import numpy as np

tokens = ["I", "love", "AI"]

# Tiny embedding vectors (chosen manually)
X = np.array([[1, 0], [0, 2], [1, 1]])  # I  # love  # AI

print("Tokens:", tokens)
print("\nEmbedding Matrix X:")
print(X)

Tokens: ['I', 'love', 'AI']

Embedding Matrix X:
[[1 0]
 [0 2]
 [1 1]]


## Step 2: Create Query, Key, and Value Weight Matrices

Normally these are learned during training. For understanding, we choose small values manually.

\[
Q = XW_Q
\]

\[
K = XW_K
\]

\[
V = XW_V
\]


In [ ]:
# Query weight matrix
WQ = np.array([[1, 0], [0, 1]])

# Key weight matrix
WK = np.array([[1, 1], [0, 1]])

# Value weight matrix
WV = np.array([[1, 0], [1, 1]])

print("WQ:\n", WQ)
print("\nWK:\n", WK)
print("\nWV:\n", WV)

WQ:
 [[1 0]
 [0 1]]

WK:
 [[1 1]
 [0 1]]

WV:
 [[1 0]
 [1 1]]


## Step 3: Compute Q, K, and V

Each embedding is projected into three different spaces.


In [ ]:
Q = X @ WQ
K = X @ WK
V = X @ WV

print("Q Matrix:\n", Q)
print("\nK Matrix:\n", K)
print("\nV Matrix:\n", V)

Q Matrix:
 [[1 0]
 [0 2]
 [1 1]]

K Matrix:
 [[1 1]
 [0 2]
 [1 2]]

V Matrix:
 [[1 0]
 [2 2]
 [2 1]]


## Step 4: Compute Raw Attention Scores

Formula:

\[
Scores = QK^T
\]

Each Query vector is compared with every Key vector using the dot product.


In [ ]:
scores = Q @ K.T

print("Attention Score Matrix:")
print(scores)

Attention Score Matrix:
[[1 0 1]
 [2 4 4]
 [2 2 3]]


### Manual Interpretation

The attention score matrix is:

```text
[[1 0 1]
 [2 4 4]
 [2 2 3]]
```

For example, the third row corresponds to the token **AI**:
- AI vs I   = 2
- AI vs love = 2
- AI vs AI  = 3

A larger score means a stronger relationship.


## Step 5: Scale the Scores

Transformer models divide by

\[
\sqrt{d_k}
\]

where `d_k` is the key dimension.

Here:
- `d_k = 2`
- `sqrt(2) ≈ 1.414`


In [ ]:
scaled_scores = scores / np.sqrt(2)

print("Scaled Scores:")
print(np.round(scaled_scores, 3))

Scaled Scores:
[[0.707 0.    0.707]
 [1.414 2.828 2.828]
 [1.414 1.414 2.121]]


**Why do we scale?**

Without scaling, the dot products become very large for high-dimensional vectors (for example, 768 dimensions in BERT). Large values cause the softmax function to become too sharp, making training unstable.

**Interview answer:**  
> We divide by \(\sqrt{d_k}\) to prevent large dot products from saturating the softmax function and causing vanishing gradients.


## Step 6: Apply Softmax

Softmax converts the raw similarity scores into probabilities.

Each row will sum to **1.0**.


In [ ]:
def softmax(x):
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


attention_weights = softmax(scaled_scores)

print("Attention Weights:")
print(np.round(attention_weights, 3))

print("\nRow sums:")
print(np.round(attention_weights.sum(axis=1), 3))

Attention Weights:
[[0.401 0.198 0.401]
 [0.108 0.446 0.446]
 [0.248 0.248 0.503]]

Row sums:
[1. 1. 1.]


### Interpretation

The attention weights are approximately:

```text
[[0.401 0.198 0.401]
 [0.108 0.446 0.446]
 [0.248 0.248 0.503]]
```

For the token **AI**, the last row means:
- 24.8% attention to **I**
- 24.8% attention to **love**
- 50.3% attention to **AI itself**


## Step 7: Compute the Final Attention Output

Final Transformer equation:

\[
Attention(Q,K,V)
=
Softmax\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
\]

The attention weights decide how much of each Value vector contributes to the final output.


In [ ]:
output = attention_weights @ V

print("Final Self-Attention Output:")
print(np.round(output, 3))

Final Self-Attention Output:
[[1.599 0.797]
 [1.892 1.337]
 [1.752 1.   ]]


### Manual Explanation for Token **AI**

Attention weights:

```text
[0.248, 0.248, 0.503]
```

Value vectors:

| Token | Value |
|--------|--------|
| I | [1,0] |
| love | [2,2] |
| AI | [2,1] |

Weighted sum:

\[
0.248[1,0] + 0.248[2,2] + 0.503[2,1]
\]

Result (approximately):

```text
[1.75, 1.00]
```

The new representation of **AI** now contains information gathered from all tokens in the sentence.


# Complete Self-Attention Flow

```text
Sentence
   │
   ▼
Tokenization
   │
   ▼
Embeddings (X)
   │
   ▼
Linear Layers
 ┌──┴───┬───┐
 ▼      ▼   ▼
 Q      K   V
 │      │   │
 └──┬───┘   │
    ▼       │
  Q × Kᵀ    │
    │       │
    ▼       │
 Divide by √dk
    │
    ▼
  Softmax
    │
    ▼
Attention Weights
    │
    ▼
Weights × V
    │
    ▼
Context-Aware Output Embeddings
```


# Interview Quick Revision

| Question | Simple Answer |
|-----------|--------------|
| What is self-attention? | Every token looks at every other token and decides how important they are. |
| What is Query? | What a token is searching for. |
| What is Key? | What information a token provides. |
| What is Value? | The information passed to the output. |
| Why QKᵀ? | To measure similarity between tokens. |
| Why divide by √dk? | To keep softmax stable. |
| Why softmax? | To convert scores into probabilities. |
| What is the final output? | New context-aware embeddings for every token. |

---

## One Formula to Remember

\[
\boxed{
Attention(Q,K,V)
=
Softmax\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
\]

Read it as:
1. Build Q, K, V.
2. Compute similarities.
3. Scale them.
4. Apply softmax.
5. Use the probabilities to combine the Value vectors.
